# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
data = spaceship.dropna().copy()

# Keep the deck: "B/15/P" becomes "B".
data["Cabin"] = data["Cabin"].str.split("/").str[0]

# X = clues. y = the answer we want to predict.
X = data.drop(columns=["PassengerId", "Name", "Transported"])
y = data["Transported"]

**Perform Train Test Split**

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=0, stratify=y
)

print("Training passengers:", len(X_train))
print("Test passengers:", len(X_test))

Training passengers: 5284
Test passengers: 1322


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

numeric_columns = X_train.select_dtypes(include="number").columns.tolist()
category_columns = X_train.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer([
    ("numbers", MinMaxScaler(), numeric_columns),
    ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
     category_columns)
])

X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [6]:
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.neighbors import KNeighborsClassifier
# from sklearn.ensemble import (
#     BaggingClassifier,
#     RandomForestClassifier,
#     GradientBoostingClassifier,
#     AdaBoostClassifier
# )

# models = {
#     "KNN baseline": KNeighborsClassifier(),

#     "Bagging": BaggingClassifier(
#         estimator=DecisionTreeClassifier(max_depth=6, random_state=42),
#         n_estimators=60, max_samples=0.8, bootstrap=True, random_state=42
#     ),

#     "Pasting": BaggingClassifier(
#         estimator=DecisionTreeClassifier(max_depth=6, random_state=42),
#         n_estimators=60, max_samples=0.8, bootstrap=False, random_state=42
#     ),
# }

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)

models = {
    "KNN baseline": KNeighborsClassifier(),

    "Bagging": BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=6, random_state=42),
        n_estimators=60, max_samples=0.8, bootstrap=True, random_state=42
    ),

    "Pasting": BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=6, random_state=42),
        n_estimators=60, max_samples=0.8, bootstrap=False, random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=60, max_depth=6, random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=60, max_depth=2, learning_rate=0.1, random_state=42
    ),

    "AdaBoost": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=2, random_state=42),
        n_estimators=60, random_state=42
    )
}


- Random Forests

In [7]:
# models = {
#      "Random Forest": RandomForestClassifier(
#         n_estimators=60, max_depth=6, random_state=42
#     ),
# }

- Gradient Boosting

In [8]:
# models = {
#      "Gradient Boosting": GradientBoostingClassifier(
#         n_estimators=60, max_depth=2, learning_rate=0.1, random_state=42
#     ),
# }

- Adaptive Boosting

In [9]:
# models = {
#      "AdaBoost": AdaBoostClassifier(
#         estimator=DecisionTreeClassifier(max_depth=2, random_state=42),
#         n_estimators=60, random_state=42
#     )
# }

Which model is the best and why?

In [10]:
from sklearn.metrics import accuracy_score, f1_score

results = []

for name, model in models.items():
    model.fit(X_train_ready, y_train)
    predictions = model.predict(X_test_ready)

    results.append({
        "Model": name,
        "Train accuracy": model.score(X_train_ready, y_train),
        "Test accuracy": accuracy_score(y_test, predictions),
        "Test F1": f1_score(y_test, predictions, pos_label=True)
    })

comparison = pd.DataFrame(results).sort_values("Test accuracy", ascending=False)
comparison.round(4)

,Model,Train accuracy,Test accuracy,Test F1
1,Bagging,0.8145,0.8011,0.8030
4,Gradient Boosting,0.8047,0.7965,0.8029
2,Pasting,0.8081,0.7950,0.7933
5,AdaBoost,0.7966,0.7920,0.7982
3,Random Forest,0.7990,0.7882,0.7816
0,KNN baseline,0.8251,0.7489,0.7470


In [11]:
#Bagging performed best by test accuracy in this comparison, scoring 0.8011 on the held-out passengers. It also scored 0.803 on F1, effectively tied with Gradient Boosting.